# 04 — 閉ループと2つの時間刻み

MuJoCoの細かい周期とMPCの遅い周期、receding horizonの関係を分離します。

**前提**: `03_mujoco_go2_plant.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
# 背景: 各章を同じ作業ディレクトリと依存関係で再実行するには、リポジトリの基準パスと外部実装の場所を最初に固定する必要がある。
# 目的: Quadruped-PyMPCをimport可能にし、acadosとヘッドレスMuJoCoの実行環境を後続セルへ引き渡す。
# ファイルシステム上の基準パスを型安全に扱うためPathを読み込む。
from pathlib import Path
# 環境変数の設定とPythonのモジュール探索パス更新に必要な標準ライブラリを読み込む。
import os, sys

# Notebookを起動した現在位置を絶対パスへ正規化し、教材全体の基準候補とする。
ROOT = Path.cwd().resolve()
# notebook_pympc直下から起動した場合だけ、リポジトリ直下へ基準を1階層戻す。
if ROOT.name == "notebook_pympc":
    # externalディレクトリを参照できるリポジトリ直下へROOTを合わせる。
    ROOT = ROOT.parent
# 上流制御実装が置かれたQuadruped-PyMPCの絶対パスを構成する。
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
# 誤った起動位置のまま進まず、依存リポジトリの欠落を具体的なパス付きで検出する。
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
# 同じパスを重複登録せず、まだimport探索対象でない場合だけ追加する。
if str(PYMPC_ROOT) not in sys.path:
    # ローカルのquadruped_pympcパッケージを通常のimport文で読めるよう探索順の先頭へ置く。
    sys.path.insert(0, str(PYMPC_ROOT))

# acados生成物・共有資源の基準位置を未設定時だけ登録し、利用者の明示設定は上書きしない。
os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
# 画面のない環境でもMuJoCoを描画可能にするため、未設定時のOpenGL backendをEGLにする。
os.environ.setdefault("MUJOCO_GL", "egl")
# 実際に採用されたworkspace基準を表示し、相対パス問題を診断できるようにする。
print("workspace :", ROOT)
# import対象となる上流実装の場所を表示し、参照しているコード版を確認可能にする。
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## Multi-rate control

- simulator: \(\Delta t_{sim}=0.002\,s\)（500 Hz）
- MPC呼出し: 100 Hz
- MPC予測刻み: \(\Delta t_{mpc}=0.02\,s\)
- horizon: \(N=12\)、予測時間 \(T=N\Delta t=0.24\,s\)

予測刻みと呼出し周期は同じとは限りません。MPCは0.24秒先まで予測しつつ、
0.01秒後には新しい観測で解き直します。

In [2]:
# 背景: simulation周期・MPC呼出し周期・予測刻みは別の時間尺度であり、設定名だけでは実際の更新間隔を判断できない。
# 目的: 現行設定から500 HzのPlant、5 simulation stepごとの100 Hz MPC、0.24 sの予測区間を導出して検算する。
# 上流実装と同じ時間刻み・horizon設定を参照するためconfigを読み込む。
from quadruped_pympc import config as cfg
# MuJoCoを1回積分する時間刻みdt_sim [s]を取得する。
sim_dt = cfg.simulation_params["dt"]
# MPCを1秒に何回呼ぶかを表す目標頻度[Hz]を取得する。
mpc_call_hz = cfg.simulation_params["mpc_frequency"]
# 1/(f_mpc*dt_sim)を整数stepへ丸め、MPC更新間のsimulation step数[-]を求める。
stride = round(1 / (mpc_call_hz * sim_dt))
# N段×予測刻みdt_mpc [s]により有限予測区間T [s]を求める。
prediction = cfg.mpc_params["horizon"] * cfg.mpc_params["dt"]
# dt_simの逆数からPlant積分頻度[Hz]を表示する。
print("simulation Hz :", 1/sim_dt)
# MPC解を保持するsimulation step数を表示し、multi-rate比を確認する。
print("MPC every     :", stride, "sim steps")
# 丸め後のstrideから実効MPC呼出し頻度[Hz]を再計算して表示する。
print("MPC call Hz   :", 1/(stride*sim_dt))
# horizon全体が覆う未来時間[s]を表示し、呼出し周期との違いを明示する。
print("horizon time  :", prediction, "s")
# 現行dt_sim=0.002 sと100 Hz設定が5 step更新になることを回帰検査する。
assert stride == 5

simulation Hz : 500.0
MPC every     : 5 sim steps
MPC call Hz   : 100.0
horizon time  : 0.24 s


## 1周期の擬似コード

1. MuJoCoから状態を読む
2. gait phaseを `dt_sim` だけ進める
3. contact sequenceとfootholdを作る
4. 5 stepごとにOCPを解き、先頭入力を保存する
5. 毎step、保存GRFと遊脚軌道からトルクを計算する
6. clipし、MuJoCoを1 step進める

この非同期性により、MPCのGRFは最大約10 ms保持されます。
高速振動が見えたとき、予測刻みだけでなく呼出し周期も調査対象です。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。